In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.healthcare_demo.pipeline_execution_log
(
    pipeline_name STRING,
    layer STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    records_processed BIGINT,
    status STRING,
    execution_time_seconds DOUBLE
)
USING DELTA;

In [0]:
%sql

INSERT INTO workspace.healthcare_demo.pipeline_execution_log
VALUES
(
'Bronze Auto Loader',
'Bronze',
current_timestamp(),
current_timestamp(),
1200520,
'SUCCESS',
45.7
);

In [0]:
%sql

INSERT INTO workspace.healthcare_demo.pipeline_execution_log
VALUES
(
'Silver Transformation',
'Silver',
current_timestamp(),
current_timestamp(),
1200520,
'SUCCESS',
31.2
);

In [0]:
%sql

INSERT INTO workspace.healthcare_demo.pipeline_execution_log
VALUES
(
'Gold Aggregation',
'Gold',
current_timestamp(),
current_timestamp(),
397408,
'SUCCESS',
12.5
);

In [0]:
%sql

SELECT *
FROM workspace.healthcare_demo.pipeline_execution_log
ORDER BY start_time DESC;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.healthcare_demo.pipeline_execution_log
(
    pipeline_name STRING,
    layer STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    records_processed BIGINT,
    status STRING,
    execution_time_seconds DOUBLE,
    error_message STRING
)
USING DELTA;

In [0]:
%sql
SHOW TABLES IN workspace.healthcare_demo
LIKE 'pipeline_execution_log';

In [0]:
%sql

DROP TABLE IF EXISTS workspace.healthcare_demo.pipeline_execution_log;

CREATE TABLE workspace.healthcare_demo.pipeline_execution_log
(
    pipeline_name STRING,
    layer STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    records_processed BIGINT,
    status STRING,
    execution_time_seconds DOUBLE,
    error_message STRING
)
USING DELTA;

In [0]:
%sql

DESCRIBE workspace.healthcare_demo.pipeline_execution_log;

In [0]:
from datetime import datetime, timezone
import time
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType,
    DoubleType
)

PIPELINE_NAME = "Silver Data Quality Check"
LAYER = "Silver"

start_time = datetime.now(timezone.utc)
start_perf = time.time()

status = "SUCCESS"
error_message = None
records_processed = 0

try:
    df = spark.table(
        "workspace.healthcare_demo.silver_patient_events"
    )

    records_processed = df.count()

except Exception as e:
    status = "FAILED"
    error_message = str(e)

end_time = datetime.now(timezone.utc)
execution_time_seconds = time.time() - start_perf

log_schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("records_processed", LongType(), False),
    StructField("status", StringType(), False),
    StructField("execution_time_seconds", DoubleType(), False),
    StructField("error_message", StringType(), True)
])

log_data = [(
    PIPELINE_NAME,
    LAYER,
    start_time,
    end_time,
    records_processed,
    status,
    execution_time_seconds,
    error_message
)]

log_df = spark.createDataFrame(
    log_data,
    schema=log_schema
)

log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.healthcare_demo.pipeline_execution_log"
    )

print("Pipeline Status:", status)
print("Records Processed:", records_processed)
print("Execution Time:", round(execution_time_seconds, 2), "seconds")

In [0]:
%sql

SELECT *
FROM workspace.healthcare_demo.pipeline_execution_log
ORDER BY start_time DESC;

In [0]:
from datetime import datetime, timezone
import time
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType,
    DoubleType
)

PIPELINE_NAME = "Failure Test"
LAYER = "Test"

start_time = datetime.now(timezone.utc)
start_perf = time.time()

status = "SUCCESS"
error_message = None
records_processed = 0

try:
    # Intentional bad table name to test error logging
    df = spark.table(
        "workspace.healthcare_demo.table_that_does_not_exist"
    )

    records_processed = df.count()

except Exception as e:
    status = "FAILED"
    error_message = str(e)

end_time = datetime.now(timezone.utc)
execution_time_seconds = time.time() - start_perf

log_schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("records_processed", LongType(), False),
    StructField("status", StringType(), False),
    StructField("execution_time_seconds", DoubleType(), False),
    StructField("error_message", StringType(), True)
])

log_data = [(
    PIPELINE_NAME,
    LAYER,
    start_time,
    end_time,
    records_processed,
    status,
    execution_time_seconds,
    error_message
)]

log_df = spark.createDataFrame(
    log_data,
    schema=log_schema
)

log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.healthcare_demo.pipeline_execution_log"
    )

print("Pipeline Status:", status)
print("Error Message:", error_message)

In [0]:
%sql
SELECT
    pipeline_name,
    layer,
    records_processed,
    status,
    execution_time_seconds,
    error_message
FROM workspace.healthcare_demo.pipeline_execution_log
ORDER BY start_time DESC;

In [0]:
%sql
SELECT *
FROM workspace.healthcare_demo.pipeline_execution_log
ORDER BY start_time DESC;